**CI twin of `ch14-naive-bayes.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
import pandas as pd
import numpy as np

df = load_csv("penguins").dropna(subset=["bill_length_mm",
                                         "flipper_length_mm"])
feats = ["flipper_length_mm", "bill_length_mm"]
Xtr, Xte, ytr, yte = train_test_split(
    df[feats], df["species"], test_size=0.25, random_state=42,
    stratify=df["species"])

nb = GaussianNB().fit(Xtr, ytr)

print("priors:", dict(zip(nb.classes_, nb.class_prior_.round(3))))
print("\nper-class feature means:")
print(pd.DataFrame(nb.theta_, index=nb.classes_, columns=feats).round(1))
print("\nper-class feature spreads (std):")
print(pd.DataFrame(np.sqrt(nb.var_), index=nb.classes_, columns=feats).round(1))

In [ ]:
import math

def gauss(x, mean, std):
    """The bell curve's height at x — how typical this value is."""
    return math.exp(-((x - mean) ** 2) / (2 * std * std)) / (
        std * math.sqrt(2 * math.pi))

probe = {"flipper_length_mm": 195.0, "bill_length_mm": 45.0}

scores = {}
for i, species in enumerate(nb.classes_):
    score = nb.class_prior_[i]                     # start from the prior
    for j, feat in enumerate(feats):
        score *= gauss(probe[feat], nb.theta_[i][j],
                       math.sqrt(nb.var_[i][j]))   # weigh each evidence
    scores[species] = score

total = sum(scores.values())
for species, s in scores.items():
    print(f"P({species:9} | evidence) = {s / total:.3f}")

sk = nb.predict_proba(pd.DataFrame([probe]))[0]
print("\nsklearn says:", dict(zip(nb.classes_, sk.round(3))))

In [ ]:
from sklearn.metrics import accuracy_score

print(f"held-out accuracy: {accuracy_score(yte, nb.predict(Xte)):.3f}")

In [ ]:
for species in ("Adelie", "Gentoo"):
    sub = df[df["species"] == species]
    r = sub["bill_length_mm"].corr(sub["flipper_length_mm"])
    print(f"{species}: bill–flipper correlation r = {r:.2f}")

In [ ]:
avg_conf = nb.predict_proba(Xte).max(axis=1).mean()
print(f"average top-class 'confidence' on held-out birds: {avg_conf:.3f}")

In [ ]:
probe = {"flipper_length_mm": 210.0, "bill_length_mm": 50.0}

scores = {}
for i, species in enumerate(nb.classes_):
    score = nb.class_prior_[i]
    for j, feat in enumerate(feats):
        score *= gauss(probe[feat], nb.theta_[i][j],
                       math.sqrt(nb.var_[i][j]))
    scores[species] = score
total = sum(scores.values())
posteriors = {sp: round(s / total, 3) for sp, s in scores.items()}

run_tests([
    ("matches sklearn", posteriors, {
        sp: round(float(p), 3) for sp, p in zip(
            nb.classes_, nb.predict_proba(pd.DataFrame([probe]))[0])
    }),
])

In [ ]:
import math

def gaussian_likelihood(x, mean, std):
    return math.exp(-((x - mean) ** 2) / (2 * std * std)) / (
        std * math.sqrt(2 * math.pi))

def posterior_scores(priors, likes):
    return [prior * math.prod(ls) for prior, ls in zip(priors, likes)]

run_tests([
    ("standard bell at its peak", round(gaussian_likelihood(0, 0, 1), 4),
     0.3989),
    ("one std out", round(gaussian_likelihood(1, 0, 1), 4), 0.242),
    ("typical beats atypical",
     gaussian_likelihood(190, 190, 7) > gaussian_likelihood(217, 190, 7),
     True),
    ("two classes scored", [round(s, 5) for s in posterior_scores(
        [0.6, 0.4], [[0.5, 0.2], [0.1, 0.9]])], [0.06, 0.036]),
    ("a zero likelihood annihilates its class", [round(s, 5) for s in
     posterior_scores([0.9, 0.1], [[0.0, 0.8], [0.5, 0.5]])], [0.0, 0.025]),
], tol=1e-3)